# 01 — Customer Segmentation with Clustering

This notebook builds a customer segmentation workflow.

The modelling goal is to discover useful behavioural groups without using predefined segment labels.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_customer_segmentation_data
from unsup_lab.evaluation import evaluate_clustering, evaluate_k_range
from unsup_lab.plotting import plot_embedding
from unsup_lab.preprocessing import scale_features
from unsup_lab.reporting import cluster_profile


## Generate customer behaviour data

In [ ]:
dataset = make_customer_segmentation_data(n_customers=2_000, random_state=42)
features = dataset.features

features.head()


The hidden labels are kept only for later sanity checks. They are not used by the clustering models.


In [ ]:
x_scaled = scale_features(features, method="standard")
x_scaled.describe().round(2)


## Compare different clustering assumptions

In [ ]:
k_results = evaluate_k_range(
    x_scaled,
    estimator_factory=lambda k: KMeans(n_clusters=k, n_init=20, random_state=42),
    k_values=[2, 3, 4, 5, 6, 7, 8],
)

k_results


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_results["k"], k_results["silhouette"], marker="o")
plt.title("Silhouette score by number of clusters")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.tight_layout()
plt.show()


## Fit selected models

In [ ]:
models = {
    "kmeans": KMeans(n_clusters=5, n_init=20, random_state=42),
    "gmm": GaussianMixture(n_components=5, random_state=42),
    "agglomerative": AgglomerativeClustering(n_clusters=5),
    "dbscan": DBSCAN(eps=1.2, min_samples=20),
}

labels_by_model = {}
for name, model in models.items():
    if hasattr(model, "fit_predict"):
        labels = model.fit_predict(x_scaled)
    else:
        labels = model.fit(x_scaled).predict(x_scaled)

    labels_by_model[name] = labels
    print(name, evaluate_clustering(x_scaled, labels))


## Visualise one selected solution

In [ ]:
pca = PCA(n_components=2, random_state=42)
embedding = pca.fit_transform(x_scaled)
selected_labels = labels_by_model["kmeans"]

plot_embedding(embedding, selected_labels, title="Customer segments in PCA space")


## Segment profiling

In [ ]:
profile = cluster_profile(features, selected_labels)
profile.round(2)


## Interpretation

The output should be converted into business-readable segment cards.

Examples:

- High-value loyal customers.
- Discount-sensitive customers.
- Low-engagement new customers.
- Frequent low-value buyers.
- At-risk previous buyers.

These names should be validated with domain experts and operational data.


## Limitations

Clusters are not natural laws. They depend on feature selection, scaling, algorithm assumptions, and the chosen number of clusters. They should guide exploration and decision-making, not be treated as causal evidence.
